# `GenVarLoader`

In [ ]:
# Automatically reload code in notebook
%load_ext autoreload
%autoreload 2

import os
os.chdir(os.path.dirname(os.path.abspath('.')))
import pandas as pd
import polars as pl
import seqpro as sp
import genvarloader as gvl

import src.utils as utils
import src.genvarloader as GVL
# import src.pyensembl as PYE

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
example_paths = GVL.prepare_example()

100%|█████████████████████████████████████| 7.26k/7.26k [00:00<00:00, 9.78MB/s]


In [9]:
bed = gvl.read_bedlike(example_paths['bed'])
bed.head()

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""chr22""",41699499,41699499,"""ENSG00000167077""",null,"""+"""
"""chr22""",42835412,42835412,"""ENSG00000100266""",null,"""-"""
"""chr22""",20858983,20858983,"""ENSG00000099940""",null,"""+"""
"""chr22""",20707691,20707691,"""ENSG00000241973""",null,"""-"""
"""chr22""",49918167,49918167,"""ENSG00000184164""",null,"""+"""


In [19]:
gvl.with_length(bed_chr, 2**17)

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""chr22""",41633963,41765035,"""ENSG00000167077""",null,"""+"""
"""chr22""",42769876,42900948,"""ENSG00000100266""",null,"""-"""
"""chr22""",20793447,20924519,"""ENSG00000099940""",null,"""+"""
"""chr22""",20642155,20773227,"""ENSG00000241973""",null,"""-"""
"""chr22""",49852631,49983703,"""ENSG00000184164""",null,"""+"""
…,…,…,…,…,…
"""chr22""",30998569,31129641,"""ENSG00000183963""",null,"""+"""
"""chr22""",50513413,50644485,"""ENSG00000100288""",null,"""-"""
"""chr22""",20933568,21064640,"""ENSG00000184436""",null,"""-"""


In [ ]:
chrom = "chr22"
ds_path = os.path.join(GVL.DATA_DIR, f"{chrom}.gvl")
bed_chr = bed.filter(pl.col("chrom")==chrom)

force=True
if not os.path.exists(ds_path) or force:
    print("Creating GVL database ==>",ds_path)
    gvl.write(
        path=ds_path,
        bed=gvl.with_length(bed_chr, 2**17), # <-- required to select sequence subsets afterwards
        variants=example_paths['pgen'],
        # max_mem=16*2**30,
        overwrite=force
    )


Creating GVL database ==> /grid/koo/home/schilder/.cache/gvl_example_data/chr22.gvl


TypeError: Expr.rolling_min() got an unexpected keyword argument 'min_samples'

In [16]:
def transform(haps, tracks):
    import numpy as np
    from einops import rearrange
    haps = rearrange(
        sp.DNA.ohe(haps), "... length alphabet -> ... alphabet length"
    ).astype(np.float32)
    return haps, tracks


ds = (
    gvl.Dataset.open(ds_path, reference=example_paths['reference'])\
    .with_seqs("haplotypes")\
    .with_tracks("read-depth")\
    .with_len(2**17)\
    .with_transform(transform)\
)

n_train = round(ds.n_samples * 0.8)
gene1_train_ds = ds.subset_to(samples=slice(0, n_train))
dl = gene1_train_ds.to_dataloader(batch_size=16, num_workers=0, shuffle=True)

FileNotFoundError: [Errno 2] No such file or directory: '/grid/koo/home/schilder/.cache/gvl_example_data/chr22.gvl/metadata.json'

## `cyvcf2`

In [ ]:
import numpy as np
from cyvcf2 import VCF

def vcf_to_genotype_matrix(vcf_file, region):
    """
    Converts a VCF file to a genotype matrix for a specific region.

    Args:
        vcf_file (str): Path to the VCF file.
        region (str): Region to extract (e.g., "chr1:1000-2000").

    Returns:
        tuple: A tuple containing:
            - genotype_matrix (numpy.ndarray): Genotype matrix (samples x variants).
            - variant_ids (list): List of variant IDs.
            - sample_ids (list): List of sample IDs.
    """
    vcf = VCF(vcf_file)
    sample_ids = vcf.samples
    genotype_matrix = []
    variant_ids = []

    for variant in vcf(region):
        variant_ids.append(variant.ID)
        genotypes = [
            sum(gt) if gt != [-1, -1] else np.nan for gt in variant.genotypes
        ]
        genotype_matrix.append(genotypes)

    vcf.close()
    return np.array(genotype_matrix).T, variant_ids, sample_ids

vcf_file_path = "/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
genotype_matrix, sample_names, variant_ids = vcf_to_genotype_matrix(vcf_file_path, "22:10664523-10684523")
print("Genotype Matrix:")
print(genotype_matrix)
print("\nSample Names:", sample_names)
print("\nVariant IDs:", variant_ids)

## `sgkit`

In [24]:
import sgkit as sg

In [25]:
ds = sg.simulate_genotype_call_dataset(n_variant=100, n_sample=50, missing_pct=.1)

In [5]:
ds[['variant_allele', 'call_genotype']]


<xarray.Dataset> Size: 10kB
Dimensions:         (variants: 100, alleles: 2, samples: 50, ploidy: 2)
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele  (variants, alleles) |S1 200B b'T' b'C' b'C' ... b'T' b'A'
    call_genotype   (variants, samples, ploidy) int8 10kB 0 0 1 0 1 ... 0 1 0 0
Attributes:
    contigs:  ['0']
    source:   sgkit-0.9.0

In [26]:
ds.keys()

KeysView(<xarray.Dataset> Size: 22kB
Dimensions:             (contigs: 1, variants: 100, alleles: 2, samples: 50,
                         ploidy: 2)
Dimensions without coordinates: contigs, variants, alleles, samples, ploidy
Data variables:
    contig_id           (contigs) <U1 4B '0'
    variant_contig      (variants) int64 800B 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    variant_position    (variants) int64 800B 0 1 2 3 4 5 ... 94 95 96 97 98 99
    variant_allele      (variants, alleles) |S1 200B b'T' b'C' ... b'T' b'A'
    sample_id           (samples) <U3 600B 'S0' 'S1' 'S2' ... 'S47' 'S48' 'S49'
    call_genotype       (variants, samples, ploidy) int8 10kB 0 0 1 0 ... 1 0 0
    call_genotype_mask  (variants, samples, ploidy) bool 10kB False ... False
Attributes:
    contigs:  ['0']
    source:   sgkit-0.9.0)

In [27]:
df = ds.to_dataframe().reset_index()
df.head()

contig_id  variant_contig  \
contigs variants alleles samples ploidy                             
0       0        0       0       0              0               0   
                                 1              0               0   
                         1       0              0               0   
                                 1              0               0   
                         2       0              0               0   

                                         variant_position variant_allele  \
contigs variants alleles samples ploidy                                    
0       0        0       0       0                      0           b'T'   
                                 1                      0           b'T'   
                         1       0                      0           b'T'   
                                 1                      0           b'T'   
                         2       0                      0           b'T'   

                                        sample_id  call_genotype  \
contigs variants alleles samples ploidy                            
0       0        0       0       0             S0              0   
                                 1             S0              0   
                         1       0             S1              1   
                                 1             S1              0   
                         2       0             S2              1   

                                         call_genotype_mask  
contigs variants alleles samples ploidy                      
0       0        0       0       0                    False  
                                 1                    False  
                         1       0                    False  
                                 1                    False  
                         2       0                    False

In [42]:
# Create a new column by combining position and allele information
df['variant_id'] = df['variant_position'].astype(str) + df['variant_allele'].astype(str) + df['call_genotype'].astype(str)

# Create a ploidy-specific sample_id column
df['sample_id_ploidy'] = df['sample_id'].astype(str) + '_' + df['ploidy'].astype(str)

In [ ]:
df.head()

,contigs,variants,alleles,samples,ploidy,contig_id,variant_contig,variant_position,variant_allele,sample_id,call_genotype,call_genotype_mask,variant_id
0,0,0,0,0,0,0,0,0,b'T',S0,0,False,0T0
1,0,0,0,0,1,0,0,0,b'T',S0,0,False,0T0
2,0,0,0,1,0,0,0,0,b'T',S1,1,False,0T1
3,0,0,0,1,1,0,0,0,b'T',S1,0,False,0T0
4,0,0,0,2,0,0,0,0,b'T',S2,1,False,0T1


Create a string representing the set of variants present in a given sample (one per ploid).

Then encode that string as an md5 hash for compression.

In [43]:
import hashlib
df_agg = df.groupby('sample_id_ploidy')['variant_id'].apply(lambda x: ','.join(x)).reset_index()
df_agg['haplotype_hash'] = df_agg['variant_id'].apply(lambda x: hashlib.md5(x.encode()).hexdigest())
df_agg.head()


,sample_id_ploidy,variant_id,haplotype_hash
0,S0_0,"0T0,0C0,1C1,1C1,2T0,2G0,3G1,3G1,4C1,4G1,5T0,5T...",6f6956e11cca55f1f743e6e9e2c83bc9
1,S0_1,"0T0,0C0,1C1,1C1,2T1,2G1,3G1,3G1,4C0,4G0,5T0,5T...",b743b59614d23b62920dad6d9da3b07d
2,S10_0,"0T1,0C1,1C1,1C1,2T-1,2G-1,3G0,3G0,4C0,4G0,5T0,...",8443e2effcff3f6caae3c255906d4a2a
3,S10_1,"0T-1,0C-1,1C1,1C1,2T0,2G0,3G1,3G1,4C1,4G1,5T1,...",719a8f29aa0f13b7cd7b4bea7c6698f2
4,S11_0,"0T0,0C0,1C0,1C0,2T0,2G0,3G0,3G0,4C1,4G1,5T0,5T...",860dcccc5edd6f6cb65af682c8498b06


Create a hashmap for each haplotype hash to a list of sample_ids.

In [47]:
# Group by haplotype_hash and get a list of sample_ids for each hash
haplotype_samples = df_agg.groupby('haplotype_hash')['sample_id_ploidy'].apply(list).reset_index()
print(haplotype_samples.shape)
haplotype_samples

(100, 2)


,haplotype_hash,sample_id_ploidy
0,03de587a4b5b60a3f447d651083c5cec,[S5_1]
1,043351dead1ad751f8b4e92e6f3a20b7,[S6_0]
2,0a09f79f2fedf6d45d534c4b46f774a6,[S25_0]
3,0a4141b62f0337efcdee50025f6ec605,[S42_1]
4,0c47c873932ee0e430ebcdf8cb08029f,[S8_0]
...,...,...
95,f268685d7979841b90f2547a0fd9d903,[S22_1]
96,f279d3537e86c67dd7e9820eb94f056e,[S38_1]
97,f5a3b9e4d38b233da2e26143b2e4371b,[S20_1]
98,fc2a2cc4e60dd60d17cab106aee9cb4b,[S35_0]


In [51]:
sg.display_genotypes(ds, max_variants=10, max_samples=10)

samples,S0,S1,S2,S3,S4,...,S45,S46,S47,S48,S49
variants,,,,,,,,,,,
0,0/0,1/0,1/0,0/1,1/0,...,1/1,0/0,1/0,0/0,1/1
1,1/1,1/0,1/.,./0,1/0,...,1/1,0/1,1/0,1/1,1/0
2,0/1,1/1,1/1,1/0,1/1,...,0/0,0/1,0/0,0/0,1/1
3,1/1,0/0,1/1,./1,0/1,...,0/1,1/0,0/1,0/.,0/.
4,1/0,0/1,0/1,0/1,0/0,...,1/0,1/1,0/0,1/.,1/0
...,...,...,...,...,...,...,...,...,...,...,...
95,1/1,1/1,./0,1/1,0/1,...,0/0,0/.,1/0,1/0,0/1
96,1/.,1/0,./0,0/1,1/0,...,0/1,1/.,0/0,1/0,0/0
97,0/1,0/0,0/0,0/1,0/0,...,0/1,0/1,1/0,1/0,0/0


In [16]:
sg.convert_call_to_index(ds).call_genotype_index.values


array([[ 0,  1,  1, ...,  1,  0,  2],
       [ 2,  1, -1, ...,  1,  2,  1],
       [ 1,  2,  2, ...,  0,  0,  2],
       ...,
       [ 1,  0,  0, ...,  1,  1,  0],
       [ 2,  0, -1, ...,  2,  1,  1],
       [ 2, -1,  0, ...,  1,  1,  0]])

## `haptools`

`haptools` provides some convenient data structures and functions for working with haplotypes. BUT it does not actually create haplotype file for you.  
Users must generate these themselves beforehand and figure out how to convert it into the haptools-specific `.hap` file format.

PLINK can generate haplotype blocks, but this is a data-driven approach (using linkage disequilibrium-clumping information from a popuatlion) that does not produce one block of a given window size (as models like SpliceAI expect).
https://www.cog-genomics.org/plink/1.9/ld#blocks

In [53]:
from haptools import data

In [54]:
vcf_file_path="/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
gt = data.GenotypesVCF(vcf_file_path) 

In [67]:
print(gt.variants[:10])
print(gt.samples[:10])

[('None', '22', 10664523, ('G', 'A')) ('None', '22', 10664525, ('G', 'C'))
 ('None', '22', 10664532, ('C', 'T')) ...
 ('None', '22', 10763475, ('T', 'C'))
 ('None', '22', 10763491, ('TG', 'T'))
 ('None', '22', 10764330, ('A', 'G'))]
('HG00096', 'HG00097', 'HG00099', 'HG00100', 'HG00101', 'HG00102', 'HG00103', 'HG00104', 'HG00105', 'HG00106')


In [62]:
genotypes = gt.read(region="22:10664523-10764523")

The data has already been loaded. Overriding.
The max_variants parameter was not specified. We have no choice but to append to an ever-growing array, which can lead to memory overuse!
[W::hts_idx_load3] The index file is older than the data file: /grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz.tbi


In [65]:
for line in gt.__iter__(region="22:10664523-10764523"):
    print(line)

[W::hts_idx_load3] The index file is older than the data file: /grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz.tbi


Record(data=array([[0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 1]], dtype=uint8), variants=array(('None', '22', 10664523, ('G', 'A')),
      dtype=[('id', '<U50'), ('chrom', '<U10'), ('pos', '<u4'), ('alleles', 'O')]))
Record(data=array([[0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 1]], dtype=uint8), variants=array(('None', '22', 10664525, ('G', 'C')),
      dtype=[('id', '<U50'), ('chrom', '<U10'), ('pos', '<u4'), ('alleles', 'O')]))
Record(data=array([[0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 1]], dtype=uint8), variants=array(('None', '22', 10664532, ('C', 'T')),
      dtype=[('id', '<U50'), ('chrom', '<U10'), ('pos', '<u4'), ('alleles', 'O')]))
Record(data=array([[0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 1]], dtype=uint8), v

In [ ]:
!haptools transform --region "22:10664523-10764523" {vcf_file_path} {vcf_file_path.replace('.vcf.gz', '.hap')}


Usage: haptools transform [OPTIONS] GENOTYPES HAPLOTYPES
Try 'haptools transform --help' for help.

Error: Invalid value for 'HAPLOTYPES': Path '/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.hap' does not exist.
